# CutMix가 원판 가림 사례에 도움이 되는가?

a03 2~14번은 실제 가림 한 시도에서 이어진 13장입니다. 기본 베이스라인과 같은 사전학습 가중치, 데이터, 100에폭을 사용하고 cutmix만 0에서 0.30으로 바꿉니다.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").exists())
sys.path.insert(0, str(root))
import pandas as pd
from wlcv.training.plate_cutmix import preview, train, compare

## 1. 학습 증강의 실제 적용 확인

기존 학습 사진 45장을 세 차례 처리합니다. attempted는 CutMix를 시도한 수, applied는 실제로 이미지 교체가 일어난 수, partial_donor는 일부만 삽입된 원판 박스 수입니다.

In [ ]:
rows = preview()
display(rows.select_dtypes(include=["number","bool"]).sum().to_frame("합계"))
print("전체 입력", len(rows))

## 2. 100에폭 학습 및 채점

실행 위치와 저장 공간을 확인한 뒤 아래 값을 True로 바꾸세요. 학습이 끝나면 같은 프레임에서 새로 찾은 원판과 잃은 원판을 함께 셉니다. 이미 학습했다면 compare에 실행 폴더를 전달할 수 있습니다.

In [ ]:
run_training = False
if run_training:
    trial = train()
    display(compare(trial))

## 3. 같은 RTX 3090에서 학습한 두 모델 비교

대조군과 실험군은 같은 사전학습 가중치에서 시작해 각각 100에폭 학습했습니다. 먼저 Mac에 저장한 평가표를 엽니다. 원본 영상부터 다시 채점하려면 아래 셀의 `rescore`를 `True`로 바꾸세요. 두 방법의 정탐 기준은 수동 박스와 IoU 0.50 이상입니다.

In [ ]:
folder = root / "outputs/plate_cutmix_3090/20261126"
review = folder / "paired_review"
rescore = False
if rescore:
    compare(folder / "runs/cutmix", control=folder / "runs/control", output=review)
if (review / "summary.csv").is_file():
    display(pd.read_csv(review / "metrics.csv"))
    display(pd.read_csv(review / "summary.csv"))
else:
    print("이 Mac에서 3090 비교 결과를 저장하면 표가 표시됩니다.")